In [7]:
import json
import os


In [8]:
data_dir = "./data"
processed_dir = os.path.join(data_dir, "processed")
base_dir = os.path.join(processed_dir, "merged")


In [9]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def load_tests_users(base_dir: str):
	users = set()

	for phase in ["Pre", "Post"]:
		phase_dir = os.path.join(base_dir, phase)

		for file in os.listdir(phase_dir):
			if file.endswith(".json"):
				path = os.path.join(phase_dir, file)
				content = load_json(path)
				users.update(content.keys())

	return users


In [10]:
def load_ndjson(path):
	lines = []

	with open(path, "r", encoding="utf-8") as f:
		for i, line in enumerate(f):
			line = line.strip()
			if line:
				try:
					lines.append(json.loads(line))
				except json.JSONDecodeError:
					print(f"Invalid JSON in {path} (line {i}): {line[:200]}")
	
	return lines

def load_traces_users(base_dir: str):
	gameplay_dir = os.path.join(base_dir, "Gameplay")

	users = set()
	for file in os.listdir(gameplay_dir):
		if file.endswith(".ndjson"):
			file_path = os.path.join(gameplay_dir, file)
			content = load_ndjson(file_path)
			for trace in content:
				user = trace.get("actor", {}).get("account", {}).get("name")
				if user:
					users.add(user)	

	return users


In [11]:
test_users = load_tests_users(base_dir)
trace_users = load_traces_users(base_dir)

print(f"Test users: {len(test_users)}")
print(f"Gameplay users: {len(trace_users)}")

valid_users = test_users & trace_users
print(f"Users in both: {len(valid_users)}")

only_in_tests = test_users - valid_users
only_in_traces = trace_users - valid_users

print("\nUsers only in tests:")
print(only_in_tests)

print("\nUsers only in gameplay:")
print(only_in_traces)



Test users: 81
Gameplay users: 81
Users in both: 79

Users only in tests:
{'684837b0e48b5a00221a37d0_ryrm', '684837b0e48b5a00221a37d0_naih'}

Users only in gameplay:
{'682b4a72c76d2e0023ed4d05_kqud', '684837b0e48b5a00221a37d0_wasa'}
